# Qwen3.5-0.8B Full Dataset Training on Colab

Train Qwen3.5-0.8B on full DialogSum (12,460 samples) with LoRA.

**Run cells 1→2→3→4→5 in order.** Crash-safe: all results saved to Google Drive.

If Colab disconnects: re-run Cell 2 + Cell 4 + Cell 5 to resume.

In [ ]:
# @title Cell 1: Install dependencies
import subprocess, sys

!pip install -q --upgrade transformers
!pip install -q datasets peft rouge-score==0.1.2 tqdm accelerate

# Build tools
!pip install -q ninja packaging wheel setuptools

# Compile causal-conv1d + flash-linear-attention with verbose output
FLA_OK = False
print("=== Compiling causal-conv1d (10-15 min, please wait) ===")
ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-v", "--no-build-isolation", "causal-conv1d"],
)
if ret.returncode == 0:
    print("=== Compiling flash-linear-attention ===")
    ret2 = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-v", "--no-build-isolation", "flash-linear-attention"],
    )
    FLA_OK = ret2.returncode == 0

if FLA_OK:
    print("SUCCESS: Linear attention CUDA kernels installed!")
else:
    print("FAILED: Training will work but linear attention layers will be slower.")

USE_FLASH_ATTN = False
print("Ready for training.")


In [ ]:
# @title Cell 2: Initialize
import os, json, random, numpy as np, time, shutil
from pathlib import Path
from typing import Dict, List

import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, __version__ as tv)
from peft import LoraConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from datasets import load_dataset
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Transformers: {tv} | Device: {device}", end="")
if torch.cuda.is_available():
    print(f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else: print()

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B", padding_side="left")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {len(tokenizer)} tokens")

ds = load_dataset("knkarthick/dialogsum")
print(f"Train: {len(ds['train'])}  Val: {len(ds['validation'])}  Test: {len(ds['test'])}")

SHORT_MAX, MEDIUM_MAX = 15, 35
BUCKET_RANGES = {"SHORT": (5,15), "MEDIUM": (16,35), "LONG": (36,999)}
LENGTH_INSTRUCTIONS = {
    "SHORT": "Write a very brief one-sentence summary of the dialogue in 5 to 15 words.",
    "MEDIUM": "Write a short summary of the dialogue in 16 to 35 words.",
    "LONG": "Write a detailed summary of the dialogue in more than 35 words.",
}
def get_length_bucket(s):
    wc = len(s.split())
    return "SHORT" if wc<=15 else "MEDIUM" if wc<=35 else "LONG"
print("Init complete.")

In [ ]:
# @title Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Cell 4: Functions (crash-safe)
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
OUTPUT_BASE = Path("/content/drive/MyDrive/qwen_colab")
MAX_INPUT, MAX_TARGET = 256, 128
PROGRESS_FILE = OUTPUT_BASE / "progress.json"

def load_progress():
    return json.loads(PROGRESS_FILE.read_text()) if PROGRESS_FILE.exists() else {}
def save_progress(p):
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    PROGRESS_FILE.write_text(json.dumps(p, indent=2))
def mark_status(name, status, **kw):
    p = load_progress(); p[name] = {"status": status, "updated": time.strftime("%Y-%m-%d %H:%M:%S"), **kw}; save_progress(p)

def build_training_pairs(dlg, summ, topic, use_len, multi):
    pairs = []
    if use_len:
        instr = LENGTH_INSTRUCTIONS[get_length_bucket(summ)]
        prompt = f"Summarize the following dialogue.\nInstruction: {instr}\nDialogue:\n{dlg}\nSummary: "
    else:
        prompt = f"Summarize the following dialogue.\nDialogue:\n{dlg}\nSummary: "
    pairs.append((prompt, summ))
    if multi:
        pairs.append((f"What is the topic of the following dialogue? Answer in a short phrase.\nDialogue:\n{dlg}\nTopic: ", topic))
    return pairs

def preprocess_batch(batch, tok, use_length_tokens=False, multitask=False):
    ids, mask, labels = [], [], []
    mx = MAX_INPUT + MAX_TARGET
    for d,s,t in zip(batch["dialogue"], batch["summary"], batch["topic"]):
        for p, tgt in build_training_pairs(d, s, t, use_length_tokens, multitask):
            pi = tok.encode(p, add_special_tokens=False)
            ti = tok.encode(tgt+tok.eos_token, add_special_tokens=False)
            if len(pi)+len(ti)>mx: pi = pi[-(mx-len(ti)):]
            fi = pi+ti
            ids.append(fi); mask.append([1]*len(fi)); labels.append([-100]*len(pi)+list(ti))
    return {"input_ids":ids, "attention_mask":mask, "labels":labels}

def load_model():
    attn = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    print(f"  Loading {MODEL_NAME} (attn={attn})...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, attn_implementation=attn)
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32,
        target_modules=["q_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM))
    model.gradient_checkpointing_disable()
    model.print_trainable_parameters()
    return model

class CausalLMCollator:
    def __init__(self, pid): self.pid = pid
    def __call__(self, fs):
        ml = max(len(f["input_ids"]) for f in fs)
        b = {"input_ids":[], "attention_mask":[], "labels":[]}
        for f in fs:
            p = ml - len(f["input_ids"])
            b["input_ids"].append(f["input_ids"]+[self.pid]*p)
            b["attention_mask"].append(f["attention_mask"]+[0]*p)
            b["labels"].append(f["labels"]+[-100]*p)
        return {k: torch.tensor(v, dtype=torch.long) for k,v in b.items()}

def find_latest_ckpt(d):
    cs = sorted(Path(d).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    return str(cs[-1]) if cs else None

def train_experiment(name, use_length_tokens=False, multitask=False, epochs=10, batch_size=16):
    od = OUTPUT_BASE / name; od.mkdir(parents=True, exist_ok=True)
    if (od / "training_complete.json").exists():
        print(f"SKIP {name}: done"); return None
    print(f"\n{'='*60}\n  {name} | len={use_length_tokens} multi={multitask} bs={batch_size}\n{'='*60}")
    print("Preprocessing...")
    kw = {"tok":tokenizer, "use_length_tokens":use_length_tokens, "multitask":multitask}
    tr = ds["train"].map(preprocess_batch, fn_kwargs=kw, batched=True, batch_size=1000, remove_columns=ds["train"].column_names, desc="Train")
    va = ds["validation"].map(preprocess_batch, fn_kwargs=kw, batched=True, batch_size=500, remove_columns=ds["validation"].column_names, desc="Val")
    print(f"  train={len(tr)} val={len(va)}")
    model = load_model()
    ckpt = find_latest_ckpt(od)
    if ckpt: print(f"  RESUMING from {ckpt}"); mark_status(name, "resuming", checkpoint=ckpt)
    else: mark_status(name, "training")
    args = TrainingArguments(output_dir=str(od), num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,
        learning_rate=5e-5, warmup_steps=100, weight_decay=0.01, max_grad_norm=1.0,
        logging_steps=50, eval_strategy="steps", eval_steps=500,
        save_strategy="steps", save_steps=500, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        bf16=True, gradient_checkpointing=False,
        dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
        processing_class=tokenizer, data_collator=CausalLMCollator(tokenizer.pad_token_id))
    print("Training...")
    t0 = time.time()
    result = trainer.train(resume_from_checkpoint=ckpt)
    elapsed = time.time() - t0
    trainer.save_model(); tokenizer.save_pretrained(od)
    loss = float(result.metrics.get("train_loss", float("nan")))
    cfg = {"experiment":name, "model":MODEL_NAME, "use_length_tokens":use_length_tokens,
        "multitask":multitask, "train_samples":len(tr), "epochs":epochs, "batch_size":batch_size,
        "gradient_accumulation_steps":4, "effective_batch_size":batch_size*4,
        "train_loss":loss, "training_time_sec":round(elapsed,1)}
    (od/"config.json").write_text(json.dumps(cfg, indent=2))
    (od/"training_complete.json").write_text(json.dumps({"status":"done","loss":loss,"elapsed":round(elapsed,1)}, indent=2))
    mark_status(name, "trained", loss=loss)
    print(f"Done. Loss={loss:.4f} Time={elapsed/60:.1f}min")
    for c in sorted(od.glob("checkpoint-*")): shutil.rmtree(c, ignore_errors=True)
    return trainer

def evaluate_experiment(name, max_samples=500):
    md = OUTPUT_BASE / name
    if not md.exists(): print(f"SKIP {name}"); return None
    cfg = json.loads((md/"config.json").read_text())
    use_len = cfg.get("use_length_tokens", False)
    attn = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    pc = PeftConfig.from_pretrained(str(md))
    base = AutoModelForCausalLM.from_pretrained(pc.base_model_name_or_path, dtype=torch.bfloat16, attn_implementation=attn)
    base.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base, str(md)).to(device).eval()
    td = ds["test"].select(range(min(max_samples, len(ds["test"]))))
    prompts, refs, buckets = [], [], []
    for r in td:
        refs.append(r["summary"]); buckets.append(get_length_bucket(r["summary"]))
        if use_len:
            inst = LENGTH_INSTRUCTIONS[get_length_bucket(r["summary"])]
            prompts.append(f"Summarize the following dialogue.\nInstruction: {inst}\nDialogue:\n{r['dialogue']}\nSummary: ")
        else:
            prompts.append(f"Summarize the following dialogue.\nDialogue:\n{r['dialogue']}\nSummary: ")
    print(f"Generating {len(prompts)} summaries for {name}...")
    preds = []
    for i in tqdm(range(0, len(prompts), 8)):
        enc = tokenizer(prompts[i:i+8], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        pl = enc["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=4, early_stopping=True,
                                  pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, do_sample=False)
        preds.extend(d.strip() for d in tokenizer.batch_decode(out[:, pl:], skip_special_tokens=True))
    from rouge_score import rouge_scorer as rs
    sc = rs.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    r1,r2,rL = [],[],[]
    for p,ref in zip(preds,refs):
        s = sc.score(ref,p); r1.append(s["rouge1"].fmeasure); r2.append(s["rouge2"].fmeasure); rL.append(s["rougeL"].fmeasure)
    res = {"experiment":name, "n":len(preds), "rouge1":round(sum(r1)/len(r1)*100,2), "rouge2":round(sum(r2)/len(r2)*100,2), "rougeL":round(sum(rL)/len(rL)*100,2)}
    if use_len:
        corr = sum(1 for p,b in zip(preds,buckets) if BUCKET_RANGES[b][0]<=len(p.split())<=BUCKET_RANGES[b][1])
        res["length_acc"] = round(corr/len(preds),4)
        for bn in ["SHORT","MEDIUM","LONG"]:
            ix = [i for i,b in enumerate(buckets) if b==bn]
            if ix: res[f"len_acc_{bn.lower()}"] = round(sum(1 for i in ix if BUCKET_RANGES[buckets[i]][0]<=len(preds[i].split())<=BUCKET_RANGES[buckets[i]][1])/len(ix),4)
    (md/"eval_results.json").write_text(json.dumps(res, indent=2))
    mark_status(name, "evaluated", rouge1=res["rouge1"], rougeL=res["rougeL"])
    print(f"\n{'='*50}\n{name} (n={res['n']})\n{'='*50}")
    print(f"  R1={res['rouge1']:.2f} R2={res['rouge2']:.2f} RL={res['rougeL']:.2f}", end="")
    if "length_acc" in res: print(f"  LenAcc={res['length_acc']*100:.1f}%", end="")
    print(f"\n  Saved to {md/'eval_results.json'}")
    return res

def run_experiment(name, use_len, multi, epochs=10, batch_size=16):
    ep = OUTPUT_BASE / name / "eval_results.json"
    if ep.exists():
        r = json.loads(ep.read_text()); print(f"SKIP {name}: RL={r['rougeL']:.2f}"); return r
    train_experiment(name, use_len, multi, epochs, batch_size)
    return evaluate_experiment(name)

print("Functions ready.")

## Cell 5: Run all experiments

Crash-safe: completed experiments skipped, interrupted ones resume.
If Colab disconnects: re-run Cell 2 + Cell 4 + Cell 5.

In [ ]:
# @title Cell 5: Run all
EXPS = [("exp0_qwen_full",False,False),("exp1_qwen_full",True,False),("exp1_multi_qwen_full",True,True)]
print("="*60+"\nPipeline start\n"+"="*60)
prog = load_progress()
for n,_,_ in EXPS: print(f"  {n}: {prog.get(n,{}).get('status','new')}")
for n,ul,mt in EXPS:
    print(f"\n{'='*60}")
    run_experiment(n, ul, mt, epochs=10, batch_size=16)
print(f"\n{'='*70}\nALL RESULTS\n{'='*70}")
for n,_,_ in EXPS:
    p = OUTPUT_BASE/n/"eval_results.json"
    if p.exists():
        r = json.loads(p.read_text())
        la = f" LenAcc={r['length_acc']*100:.1f}%" if "length_acc" in r else ""
        print(f"  {n}: R1={r['rouge1']:.2f} R2={r['rouge2']:.2f} RL={r['rougeL']:.2f}{la}")
    else: print(f"  {n}: NOT DONE")

In [ ]:
# @title Cell 6: Check progress
prog = load_progress()
for n,info in prog.items(): print(f"  {n}: {info.get('status','?')}")
print("\nDrive files:")
for n in ["exp0_qwen_full","exp1_qwen_full","exp1_multi_qwen_full"]:
    d = OUTPUT_BASE/n
    if d.exists(): print(f"  {n}/: {', '.join(sorted(f.name for f in d.iterdir() if not f.name.startswith('checkpoint')))}")
    else: print(f"  {n}/: not found")

In [ ]:
# @title Cell 7: Download results
!cd /content/drive/MyDrive/qwen_colab && zip -r results.zip exp0_qwen_full/eval_results.json exp0_qwen_full/config.json exp1_qwen_full/eval_results.json exp1_qwen_full/config.json exp1_multi_qwen_full/eval_results.json exp1_multi_qwen_full/config.json progress.json 2>/dev/null
from google.colab import files
files.download('/content/drive/MyDrive/qwen_colab/results.zip')

In [ ]:
# @title Cell 8: Download models (optional, large)
!cd /content/drive/MyDrive/qwen_colab && zip -r models.zip exp0_qwen_full exp1_qwen_full exp1_multi_qwen_full -x "*/checkpoint-*"
files.download('/content/drive/MyDrive/qwen_colab/models.zip')